# 8-3절 연습 문제 풀이

이 노트북은 8-3절 연습 문제(8-8 ~ 8-11)의 풀이 예시다.

- 본문 예제 코드는 `code_examples/ch08/08-03_example.ipynb`를 참고한다.
- 학습 환경과 데이터로더는 본문 예제와 같게 맞춰 결과를 그대로 견줄 수 있게 했다.
- 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

## 공통 준비 — 환경 설정, 데이터로더, 기준 모델, 학습 함수

In [1]:
# 환경 설정
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

common.set_korean_plot_env()
viz.configure(save_grayscale=False)

SEED = 42
common.set_seed(SEED, deterministic=True)
device = common.get_device()

CUDA를 사용합니다.


In [2]:
# CIFAR-10 데이터로더 (본문 예제와 동일)
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

data_root = '../../download'
CIFAR_MEAN = (0.5, 0.5, 0.5)
CIFAR_STD = (0.5, 0.5, 0.5)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

TRAIN_SIZE, VALID_SIZE = 40000, 10000
BATCH_SIZE = 64

full_train_set = datasets.CIFAR10(root=data_root, train=True,
                                  download=True, transform=train_transform)
full_eval_set = datasets.CIFAR10(root=data_root, train=True,
                                 download=True, transform=transform)
test_set = datasets.CIFAR10(root=data_root, train=False,
                            download=True, transform=transform)

common.set_seed(SEED, deterministic=True)
train_set, valid_set = random_split(full_train_set, [TRAIN_SIZE, VALID_SIZE])
common.set_seed(SEED, deterministic=True)
train_evaluate_set, _ = random_split(full_eval_set, [TRAIN_SIZE, VALID_SIZE])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)
train_evaluate_loader = DataLoader(train_evaluate_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print(f'훈련 {len(train_set)}, 검증 {len(valid_set)}, 평가 {len(test_set)}')

훈련 40000, 검증 10000, 평가 10000


In [3]:
# 기준 블록과 모델 ([코드 8-5], [코드 8-6]과 같음)
import torch.nn as nn


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Identity()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.activation = nn.ReLU()

    def forward(self, x):
        out = self.main_path(x) + self.shortcut(x)
        out = self.activation(out)
        return out


class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ResidualBlock(3, 32),                   # (B, 3, 32, 32) -> (B, 32, 32, 32)
            ResidualBlock(32, 32),
            nn.MaxPool2d(kernel_size=2, stride=2),  #                -> (B, 32, 16, 16)
            ResidualBlock(32, 64),
            ResidualBlock(64, 64),
            nn.MaxPool2d(kernel_size=2, stride=2),  #                -> (B, 64, 8, 8)
            ResidualBlock(64, 128),
            nn.MaxPool2d(kernel_size=2, stride=2)   #                -> (B, 128, 4, 4)
        )
        self.pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pooling(x)
        x = self.classifier(x)
        return x

In [4]:
# 학습에 사용하는 함수 (본문 예제와 동일)
import copy

import torch.optim as optim


def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    loss_sum, sample_size = 0.0, 0
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * inputs.size(0)
        sample_size += inputs.size(0)
    return loss_sum / sample_size


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct_size, sample_size = 0.0, 0, 0
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        loss_sum += criterion(outputs, labels).item() * inputs.size(0)
        correct_size += (outputs.argmax(dim=1) == labels).sum().item()
        sample_size += inputs.size(0)
    return loss_sum / sample_size, correct_size / sample_size


def run_training(model_factory, name, epochs=200, patience=10, lr=1e-3):
    common.set_seed(SEED, deterministic=True)
    model = model_factory()
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    log = common.EpochLogger(epochs, target_rows=epochs)
    best_valid_loss, best_epoch, best_params, patience_counter = float('inf'), -1, None, 0
    print(f'{name} 학습')
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)
        log.row(epoch, train_loss, valid_loss, valid_acc * 100)
        if valid_loss < best_valid_loss:
            best_valid_loss, best_epoch = valid_loss, epoch
            best_params = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    if best_params is not None:
        print(f'최적 에포크({best_epoch}, 최소 검증 손실 {best_valid_loss:.4f})의 파라미터로 복원')
        model.load_state_dict(best_params)
    train_loss, train_acc = evaluate(model, train_evaluate_loader, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    result = {'name': name, 'best_epoch': best_epoch,
              'train_loss': train_loss, 'test_loss': test_loss,
              'train_acc': train_acc * 100, 'test_acc': test_acc * 100,
              'params': common.count_params(model)}
    print(f"{name}: 최적 에포크 {best_epoch}, 훈련 손실 {train_loss:.4f} / "
          f"평가 손실 {test_loss:.4f}, 훈련 정확도 {train_acc * 100:.2f}% / "
          f"평가 정확도 {test_acc * 100:.2f}%")
    return model, result


def print_results(results, keys):
    print(f"{'모델':<26}{'최적 에포크':>10}{'훈련 손실':>10}{'평가 손실':>10}"
          f"{'훈련 정확도':>12}{'평가 정확도':>12}{'파라미터':>12}")
    print('-' * 92)
    for k in keys:
        r = results[k]
        print(f"{r['name']:<26}{r['best_epoch']:>10}{r['train_loss']:>10.4f}"
              f"{r['test_loss']:>10.4f}{r['train_acc']:>11.2f}%{r['test_acc']:>11.2f}%"
              f"{r['params']:>12,}")


EPOCHS = 200
PATIENCE = 10
LR = 1e-3
results = {}

---

## 연습 문제 8-8

> `ResidualBlock` 클래스에 포함된 합성곱 계층은 스트라이드 인자(`stride`)를 생략해 기본값(1)을
> 사용한다. 기본값 대신 생성자가 전달받은 값을 사용하도록 `ResidualBlock` 클래스를 수정해 보자.
> 수정 후, 스트라이드 인자를 2로 바꾸면 특징 지도의 크기가 변하는지 확인해 보자.
>
> 힌트: 주 경로의 크기에 따라 지름길 경로의 합성곱 계층도 수정이 필요하다.

In [5]:
# 스트라이드 인자를 받도록 수정한 잔차 블록
class StridedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # 주 경로: 첫 번째 합성곱 계층에만 스트라이드를 적용해 특징 지도를 줄인다
        #     두 번째 합성곱 계층까지 줄이면 한 블록에서 두 번 줄어든다
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        # 지름길 경로: 채널 수가 바뀌거나 특징 지도의 크기가 바뀌면 어댑터가 필요하다
        #     스트라이드가 1이 아니면 1x1 합성곱에도 같은 스트라이드를 줘야 형태가 맞는다
        self.shortcut = nn.Identity()
        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.activation = nn.ReLU()

    def forward(self, x):
        out = self.main_path(x) + self.shortcut(x)
        out = self.activation(out)
        return out

In [6]:
# 특징 지도의 크기 변화 확인
sample = torch.randn(4, 32, 16, 16)

cases = [
    ('stride=1, 채널 유지', StridedResidualBlock(32, 32, stride=1)),
    ('stride=2, 채널 유지', StridedResidualBlock(32, 32, stride=2)),
    ('stride=1, 채널 변경', StridedResidualBlock(32, 64, stride=1)),
    ('stride=2, 채널 변경', StridedResidualBlock(32, 64, stride=2)),
]
print(f'입력 형태: {tuple(sample.shape)}')
print(f"{'설정':<22}{'출력 형태':>22}{'지름길 경로':>18}")
print('-' * 62)
for label, block in cases:
    block.eval()
    with torch.no_grad():
        out = block(sample)
    kind = '항등 함수' if isinstance(block.shortcut, nn.Identity) else '어댑터(1x1 합성곱)'
    print(f'{label:<22}{str(tuple(out.shape)):>22}{kind:>18}')

입력 형태: (4, 32, 16, 16)
설정                                     출력 형태            지름길 경로
--------------------------------------------------------------
stride=1, 채널 유지              (4, 32, 16, 16)             항등 함수
stride=2, 채널 유지                (4, 32, 8, 8)      어댑터(1x1 합성곱)
stride=1, 채널 변경              (4, 64, 16, 16)      어댑터(1x1 합성곱)
stride=2, 채널 변경                (4, 64, 8, 8)      어댑터(1x1 합성곱)


In [7]:
# 힌트가 가리키는 지점 확인 - 지름길 경로에 스트라이드를 주지 않으면 어떻게 되는가
class BrokenStridedBlock(nn.Module):
    """지름길 경로의 1x1 합성곱에 스트라이드를 빠뜨린 잘못된 구현"""

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        # stride를 넘기지 않았다 (잘못된 부분)
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        return self.main_path(x) + self.shortcut(x)


broken = BrokenStridedBlock(32, 64, stride=2)
broken.eval()
try:
    with torch.no_grad():
        broken(sample)
except RuntimeError as error:
    print('예외 발생:', error)

예외 발생: The size of tensor a (8) must match the size of tensor b (16) at non-singleton dimension 3


### 풀이 해설 — 연습 문제 8-8

수정해야 할 곳은 두 군데다.

1. **주 경로의 첫 번째 합성곱 계층에만 스트라이드를 준다.** 두 합성곱 계층 모두에 주면 한
   블록에서 특징 지도가 네 배로 줄어든다. ResNet도 블록마다 한 번씩만 줄인다.
2. **지름길 경로의 1x1 합성곱에도 같은 스트라이드를 준다.** 이것이 문제의 힌트가 가리키는
   지점이다. 주 경로만 줄이면 두 경로의 특징 지도 크기가 달라져 더할 수 없고, 위 셀처럼
   `RuntimeError`가 난다.

한 가지 더 있다. 원래 `ResidualBlock`은 **채널 수가 바뀔 때만** 어댑터를 두지만, 스트라이드를
도입하면 **채널 수가 같아도 크기가 바뀌면** 어댑터가 필요하다. 조건을
`in_channels != out_channels`에서 `in_channels != out_channels or stride != 1`로 넓혀야 한다.

이는 본문 각주 16이 "합성곱 계층의 패딩이나 스트라이드 인자의 값에 따라 출력 특징 지도의 크기가
바뀌는 경우도 마찬가지의 어댑터가 필요하다"고 미리 짚어 둔 내용과 정확히 맞물린다.

`stride=2`를 주면 최대 풀링 계층 없이도 특징 지도가 절반으로 줄어든다. 실제 ResNet이 바로 이
방식으로 크기를 줄이며, 8-3절 예제가 최대 풀링을 따로 두는 것과 다른 점이다.

---

## 연습 문제 8-9

> `MiniResNet` 모델은 합성곱 계층 두 개마다 지름길을 하나씩 둔 구조다. 합성곱 계층 하나마다
> 지름길을 하나씩 두도록 클래스를 수정한 후 결과를 확인해 보자.

In [8]:
# 합성곱 계층 하나마다 지름길을 둔 잔차 블록
class SingleConvResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # 주 경로에 합성곱 계층이 하나뿐이다 (활성화 함수가 들어갈 자리가 없다)
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Identity()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(self.main_path(x) + self.shortcut(x))


class MiniResNetSingle(nn.Module):
    """MiniResNet과 합성곱 계층 수(10개)와 채널 흐름은 같고, 지름길만 두 배로 늘린 모델"""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            SingleConvResidualBlock(3, 32),
            SingleConvResidualBlock(32, 32),
            SingleConvResidualBlock(32, 32),
            SingleConvResidualBlock(32, 32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            SingleConvResidualBlock(32, 64),
            SingleConvResidualBlock(64, 64),
            SingleConvResidualBlock(64, 64),
            SingleConvResidualBlock(64, 64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            SingleConvResidualBlock(64, 128),
            SingleConvResidualBlock(128, 128),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.pooling(self.features(x)))


for cls in (MiniResNet, MiniResNetSingle):
    m = cls()
    convs = sum(1 for mod in m.modules() if isinstance(mod, nn.Conv2d)
                and mod.kernel_size == (3, 3))
    shortcuts = sum(1 for mod in m.modules()
                    if isinstance(mod, (ResidualBlock, SingleConvResidualBlock)))
    print(f'{cls.__name__}: 3x3 합성곱 {convs}개, 지름길 {shortcuts}개, '
          f'파라미터 {common.count_params(m):,}개')

MiniResNet: 3x3 합성곱 10개, 지름길 5개, 파라미터 392,074개
MiniResNetSingle: 3x3 합성곱 10개, 지름길 10개, 파라미터 392,074개


In [9]:
model_resnet, results['MiniResNet'] = run_training(MiniResNet, 'MiniResNet', EPOCHS, PATIENCE, LR)

MiniResNet 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/200       1.3179       1.1498       59.69%     0:14


  2/200       0.8910       0.8155       71.61%     0:30


  3/200       0.7253       0.7454       73.97%     0:45


  4/200       0.6365       0.8472       70.53%     1:01


...

 50/200       0.1132       0.4006       88.23%    12:19


 51/200       0.1068       0.3956       88.69%    12:33


 52/200       0.1054       0.4325       88.22%    12:48


 53/200       0.1066       0.4011       88.80%    13:03
최적 에포크(43, 최소 검증 손실 0.3728)의 파라미터로 복원


MiniResNet: 최적 에포크 43, 훈련 손실 0.1000 / 평가 손실 0.4237, 훈련 정확도 96.51% / 평가 정확도 88.48%


In [10]:
model_single, results['Single'] = run_training(
    MiniResNetSingle, 'MiniResNet(지름길 2배)', EPOCHS, PATIENCE, LR)

MiniResNet(지름길 2배) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/200       1.3295       1.1401       59.41%     0:15


  2/200       0.9146       0.9628       65.41%     0:29


  3/200       0.7760       0.8374       70.44%     0:45


  4/200       0.6802       0.9484       67.87%     0:59


...

 40/200       0.1601       0.4383       86.68%     9:58


 41/200       0.1643       0.3896       87.92%    10:13


 42/200       0.1522       0.4082       88.04%    10:27


 43/200       0.1518       0.3877       88.08%    10:42
최적 에포크(33, 최소 검증 손실 0.3685)의 파라미터로 복원


MiniResNet(지름길 2배): 최적 에포크 33, 훈련 손실 0.1599 / 평가 손실 0.4160, 훈련 정확도 94.36% / 평가 정확도 87.61%


In [11]:
print_results(results, ['MiniResNet', 'Single'])

모델                            최적 에포크     훈련 손실     평가 손실      훈련 정확도      평가 정확도        파라미터
--------------------------------------------------------------------------------------------
MiniResNet                        43    0.1000    0.4237      96.51%      88.48%     392,074
MiniResNet(지름길 2배)                33    0.1599    0.4160      94.36%      87.61%     392,074


### 풀이 해설 — 연습 문제 8-9

본문은 이 구조의 문제를 미리 말해 두었다.

> 합성곱 계층 하나마다 지름길 경로가 있으면 잔차가 지나치게 단순해져 선형 함수가 되어 버린다.

주 경로에 합성곱 계층이 하나뿐이면 그 안에는 `합성곱 -> 배치 정규화`만 남고 **활성화 함수가
들어갈 자리가 없다.** 합성곱과 배치 정규화는 둘 다 선형 변환이므로, 주 경로 전체가 하나의
선형 변환으로 수렴한다. 즉 각 블록이 학습하는 잔차의 표현력이 크게 떨어진다.

비선형성이 아주 사라지는 것은 아니다. 블록 끝의 `ReLU`는 그대로 있으므로 블록 사이에는
비선형성이 남는다. 하지만 **잔차 자체가 선형**이어서, "입력을 기준으로 삼아 복잡한 수정분을
학습한다"는 잔차 학습의 취지가 옅어진다.

이 문제는 본문이 답을 먼저 말하고 독자가 확인하는 구조다. 그래서 단순한 확인 작업으로 끝나기
쉬운데, 아래 문제 검토에 이 점을 적었다. 실행 결과는 위 표로 확인한다.

---

## 연습 문제 8-10 [도전 문제]

> `MiniResNet`은 잔차 블록 다섯 개로 이뤄진 비교적 얕은 모델이다. 풀링 계층 사이에 입력과 출력
> 채널 수가 같은 잔차 블록(예: `ResidualBlock(32, 32)`)을 여러 개 더 쌓아 훨씬 깊은 모델을 만들고,
> [연습 문제 8-2], [연습 문제 8-6]에서처럼 학습시킨 뒤 다음 질문에 답해 보자.
> - 계층을 깊게 쌓아도 학습이 안정적으로 이뤄지는지, 그리고 분류 성능은 어떻게 달라지는지
>   `MiniResNet`과 비교해 확인해 보자.
> - 확인한 결과를 바탕으로 모델의 성능을 높이기 위한 방법을 제시해 보자.

원고는 "잔차 블록 **세 개**"라고 적었지만 [코드 8-6]의 `MiniResNet`에는 잔차 블록이
**다섯 개** 있다(2단계 보고서 항목 32). 여기서는 다섯 개를 기준으로 푼다.

In [12]:
# 채널 수가 같은 잔차 블록을 단계마다 더 쌓아 깊게 만든 모델
class DeepMiniResNet(nn.Module):
    def __init__(self, blocks_per_stage=3, num_classes=10):
        """blocks_per_stage: 각 단계에서 채널 수를 유지하며 추가로 쌓을 잔차 블록 수"""
        super().__init__()
        layers = []
        for in_c, out_c in ((3, 32), (32, 64), (64, 128)):
            layers.append(ResidualBlock(in_c, out_c))
            for _ in range(blocks_per_stage):
                layers.append(ResidualBlock(out_c, out_c))
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.features = nn.Sequential(*layers)
        self.pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.pooling(self.features(x)))


for n in (1, 3):
    m = DeepMiniResNet(blocks_per_stage=n)
    blocks = sum(1 for mod in m.modules() if isinstance(mod, ResidualBlock))
    convs = sum(1 for mod in m.modules()
                if isinstance(mod, nn.Conv2d) and mod.kernel_size == (3, 3))
    print(f'DeepMiniResNet(blocks_per_stage={n}): 잔차 블록 {blocks}개, '
          f'3x3 합성곱 {convs}개, 파라미터 {common.count_params(m):,}개')

DeepMiniResNet(blocks_per_stage=1): 잔차 블록 6개, 3x3 합성곱 12개, 파라미터 687,498개
DeepMiniResNet(blocks_per_stage=3): 잔차 블록 12개, 3x3 합성곱 24개, 파라미터 1,463,434개


In [13]:
model_deep, results['Deep'] = run_training(
    lambda: DeepMiniResNet(blocks_per_stage=3), 'MiniResNet 깊게(블록 12)',
    EPOCHS, PATIENCE, LR)

MiniResNet 깊게(블록 12) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/200       1.4364       1.0998       60.25%     0:18


  2/200       0.9540       0.9431       65.68%     0:35


  3/200       0.7596       0.7837       72.18%     0:53


  4/200       0.6493       0.8146       71.53%     1:10


...

 31/200       0.1282       0.3872       88.58%     9:03


 32/200       0.1229       0.3411       89.58%     9:21


 33/200       0.1173       0.3600       89.49%     9:44


 34/200       0.1117       0.4108       88.81%    10:05
최적 에포크(24, 최소 검증 손실 0.3387)의 파라미터로 복원


MiniResNet 깊게(블록 12): 최적 에포크 24, 훈련 손실 0.1220 / 평가 손실 0.3772, 훈련 정확도 95.61% / 평가 정확도 88.89%


In [14]:
print_results(results, ['MiniResNet', 'Deep'])

모델                            최적 에포크     훈련 손실     평가 손실      훈련 정확도      평가 정확도        파라미터
--------------------------------------------------------------------------------------------
MiniResNet                        43    0.1000    0.4237      96.51%      88.48%     392,074
MiniResNet 깊게(블록 12)              24    0.1220    0.3772      95.61%      88.89%   1,463,434


### 풀이 해설 — 연습 문제 8-10

첫 번째 질문(학습이 안정적으로 이뤄지는가)에 대한 답은 실행 결과로 확인한다. 핵심은
**깊이를 두 배 넘게 늘려도 학습이 무너지지 않는다**는 점이다. [연습 문제 8-2]에서 배치 정규화
없이 계층을 늘렸을 때, [연습 문제 8-6]에서 배치 정규화만으로 늘렸을 때와 견주면 잔차 학습이
무엇을 해결했는지가 드러난다.

두 번째 질문(성능을 높이는 방법)에 대해 제시할 수 있는 방향은 다음과 같다.

1. **데이터가 한계다.** CIFAR-10은 32x32 해상도에 훈련 샘플이 4만 개뿐이다. 깊이를 늘려 표현력을
   키워도 학습할 내용 자체가 늘지 않으므로, 어느 지점부터는 깊이가 성능으로 이어지지 않는다.
   더 큰 데이터셋이나 더 강한 데이터 증강이 먼저다.
2. **학습률 조정(스케줄링)** — 깊은 모델일수록 고정 학습률로는 최적점 근처에서 맴돈다. 학습이
   진행되면서 학습률을 낮추면 같은 구조로도 더 낮은 손실에 닿는다.
3. **병목 구조** — [연습 문제 8-11]이 다루는 방식으로, 같은 파라미터 예산에서 더 깊게 쌓을 수 있다.
4. **가중치 초기화** — 학습 노트가 소개한 카이밍 허 초기화처럼 ReLU에 맞춘 초기화가 깊은 모델의
   학습 초반을 안정시킨다.

특히 1번을 짚는 것이 중요하다. 8-3절 본문도 "CIFAR-10 데이터셋은 합성곱 계층의 수를 늘린 더
깊은 신경망에 적용하기에는 적합하지 않다"고 밝히고 있어, 이 문제의 답과 본문이 이어진다.

---

## 연습 문제 8-11 [도전 문제]

> 각주 14에서 언급한 병목 구조는 ResNet-50, ResNet-101, ResNet-152 같은 깊은 ResNet이 연산량을
> 줄이려고 사용하는 잔차 블록의 변형이다. 다음 세 단계로 구성된 잔차를 만든다.
> - 1x1 합성곱: 채널 수를 압축한다(출력 채널의 1/4이 표준).
> - 3x3 합성곱: 압축된 채널에서 특징을 추출한다.
> - 1x1 합성곱: 채널 수를 원래 출력 채널 수로 복원한다.
>
> 이 구조를 가진 잔차 블록 `BottleneckBlock` 클래스를 정의하자. 그런 다음 `MiniResNet`의
> `ResidualBlock`을 `BottleneckBlock`으로 교체한 새 모델을 만들어, 다음 관점에서 `MiniResNet`과
> 비교해 보자.
> - 전체 파라미터 수와 메모리 사용량
> - 모델의 성능

In [15]:
# 병목 구조 잔차 블록
class BottleneckBlock(nn.Module):
    def __init__(self, in_channels, out_channels, compression=4):
        super().__init__()
        mid_channels = max(out_channels // compression, 1)
        # 주 경로: 1x1(압축) -> 3x3(특징 추출) -> 1x1(복원)
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(),
            nn.Conv2d(mid_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Identity()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(self.main_path(x) + self.shortcut(x))


class MiniResNetBottleneck(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            BottleneckBlock(3, 32),
            BottleneckBlock(32, 32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            BottleneckBlock(32, 64),
            BottleneckBlock(64, 64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            BottleneckBlock(64, 128),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.pooling(self.features(x)))

In [16]:
# 파라미터 수와 메모리 사용량 비교
from torchinfo import summary

MB = 10 ** 6
print(f"{'항목':<26}{'MiniResNet':>14}{'병목 구조':>16}")
print('-' * 56)
infos = [summary(cls(), input_size=(64, 3, 32, 32), device='cpu', verbose=0)
         for cls in (MiniResNet, MiniResNetBottleneck)]
rows = [
    ('파라미터 수(개)', [f'{i.total_params:,}' for i in infos]),
    ('파라미터 메모리(MB)', [f'{i.total_param_bytes / MB:.2f}' for i in infos]),
    ('순전파/역전파 메모리(MB)', [f'{i.total_output_bytes / MB:.2f}' for i in infos]),
    ('연산량(mult-adds, G)', [f'{i.total_mult_adds / 1e9:.2f}' for i in infos]),
]
for label, values in rows:
    print(f'{label:<26}{values[0]:>14}{values[1]:>16}')

항목                            MiniResNet           병목 구조
--------------------------------------------------------
파라미터 수(개)                        392,074          38,530
파라미터 메모리(MB)                        1.57            0.15
순전파/역전파 메모리(MB)                   276.83          222.30
연산량(mult-adds, G)                   4.96            0.40


In [17]:
model_bottleneck, results['Bottleneck'] = run_training(
    MiniResNetBottleneck, 'MiniResNet(병목 구조)', EPOCHS, PATIENCE, LR)

MiniResNet(병목 구조) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/200       1.5722       1.3766       49.93%     0:15


  2/200       1.2461       1.1433       58.45%     0:31


  3/200       1.1015       1.0833       61.45%     0:47


  4/200       1.0139       1.0457       63.05%     1:03


...

 86/200       0.3941       0.4793       83.42%    22:52


 87/200       0.3910       0.4841       83.52%    23:10


 88/200       0.3933       0.4897       83.28%    23:28


 89/200       0.3878       0.4931       82.63%    23:45
최적 에포크(79, 최소 검증 손실 0.4655)의 파라미터로 복원


MiniResNet(병목 구조): 최적 에포크 79, 훈련 손실 0.3571 / 평가 손실 0.5169, 훈련 정확도 87.67% / 평가 정확도 82.97%


In [18]:
print_results(results, ['MiniResNet', 'Bottleneck'])

모델                            최적 에포크     훈련 손실     평가 손실      훈련 정확도      평가 정확도        파라미터
--------------------------------------------------------------------------------------------
MiniResNet                        43    0.1000    0.4237      96.51%      88.48%     392,074
MiniResNet(병목 구조)                 79    0.3571    0.5169      87.67%      82.97%      38,530


### 풀이 해설 — 연습 문제 8-11

실행 결과는 다음과 같다.

| 항목 | `MiniResNet` | 병목 구조 | 변화 |
|---|---|---|---|
| 파라미터 수 | 392,074 | **38,530** | **10분의 1** |
| 파라미터 메모리(MB) | 1.57 | 0.15 | 10분의 1 |
| 순전파/역전파 메모리(MB) | 276.83 | 222.30 | 20% 감소 |
| 연산량(mult-adds, G) | 4.96 | **0.40** | **12분의 1** |
| 최적 에포크 | 43 | 79 | 1.8배 |
| 평가 정확도 | **88.48%** | 82.97% | −5.51%p |

**파라미터와 연산량의 절감이 압도적이다.** 채널 수가 32, 64, 128로 작아서 이득이 크지 않을 것
같지만, 절감은 **비율로 작동하므로** 채널 수와 무관하게 크다. 입력과 출력이 모두 64채널인
블록으로 계산해 보면 분명하다.

```
원래 잔차 블록: 64 x 64 x 9 x 2                       = 73,728개
병목 블록     : 64 x 16 + 16 x 16 x 9 + 16 x 64        =  4,352개   (17분의 1)
```

3x3 합성곱을 4분의 1로 압축된 채널에서만 돌리는데, 3x3의 파라미터는 `입력 채널 x 출력 채널`에
비례하므로 **양쪽을 모두 4분의 1로 줄이면 16분의 1이 된다.** 1x1 두 개를 더하는 비용은 그에 비해
작다.

**반면 순전파/역전파 메모리는 20%밖에 줄지 않았다.** 특징 지도가 차지하는 메모리는
`채널 수 x 가로 x 세로`에 비례하는데, 병목 블록도 **출력 채널 수는 그대로**이고 해상도도 같기
때문이다. 압축된 것은 블록 **안쪽**뿐이다. 연습 문제 8-1에서 본 것과 같은 교훈이다 —
**파라미터 수와 메모리 사용량은 따로 움직인다.**

**성능은 5.51%p 떨어졌다.** 파라미터가 10분의 1이 되었으니 모델의 용량도 그만큼 줄었다.
당연한 결과다.

**여기서 병목 구조의 진짜 쓰임새가 드러난다.** ResNet-50은 병목 구조로 **아낀 예산을 깊이에
재투자한다.** 같은 파라미터 예산으로 블록을 훨씬 많이 쌓는 것이다. 그런데 이 연습 문제는
`MiniResNet`의 블록 수를 그대로 두고 블록만 바꿨으므로, **아낀 예산을 쓰지 않고 그냥
작아진 모델**이 된다.

따라서 이 문제의 결과를 "병목 구조는 성능을 떨어뜨린다"로 읽으면 안 된다. 정확한 독법은
**"병목 구조는 같은 깊이를 10분의 1 파라미터로 만든다"**이고, 남은 질문은 **"그러면 아낀 예산으로
얼마나 더 깊게 쌓을 수 있는가"**다. [연습 문제 8-10]과 이어서 생각해 볼 만하다.

최적 에포크가 43에서 79로 늘어난 것도 눈여겨보자. 모델이 작아 한 에포크에서 배우는 양이
줄었고, 그만큼 수렴이 느려졌다.
